
# Practical Encoding Techniques on Telco Customer Churn Dataset

## Objective
This notebook demonstrates multiple encoding techniques used in machine learning preprocessing.

We will:
- Load and explore the Telco Customer Churn dataset
- Perform basic preprocessing
- Split the data into train and test sets
- Apply different encoding methods
- Compare how each encoding affects the dataset

Encoding techniques covered:
- Label Encoding
- One Hot Encoding
- Frequency Encoding
- Target Encoding


## 1. Import Required Libraries

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


## 2. Load Dataset

In [2]:

df = pd.read_csv("Telco-Customer-Churn.csv")
df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Dataset Inspection

In [3]:
df.shape

(7043, 21)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 



## 4. Basic Preprocessing
Remove identifier column and fix datatype issues.


In [5]:

df.drop("customerID", axis=1, inplace=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


C:\Users\chour\AppData\Local\Temp\ipykernel_22208\2602343900.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


## 5. Separate Features and Target

In [6]:

X = df.drop("Churn", axis=1)
y = df["Churn"].map({"Yes":1, "No":0})



## 6. Train Test Split
We split before encoding to avoid data leakage.


In [7]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



## 7. Label Encoding (Binary Features)
Used for features with only two categories.


In [8]:

binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']

X_train_label = X_train.copy()
X_test_label = X_test.copy()

le = LabelEncoder()

for col in binary_cols:
    X_train_label[col] = le.fit_transform(X_train_label[col])
    X_test_label[col] = le.transform(X_test_label[col])



## 8. One Hot Encoding
Suitable for nominal categorical features without inherent order.


In [9]:

nominal_cols = ['InternetService','Contract','PaymentMethod','MultipleLines','gender']

X_train_ohe = pd.get_dummies(X_train, columns=nominal_cols, drop_first=True)
X_test_ohe = pd.get_dummies(X_test, columns=nominal_cols, drop_first=True)



## 9. Frequency Encoding
Replaces categories with their occurrence counts.


In [10]:

X_train_freq = X_train.copy()
X_test_freq = X_test.copy()

freq_map = X_train_freq['PaymentMethod'].value_counts()

X_train_freq['PaymentMethod'] = X_train_freq['PaymentMethod'].map(freq_map)
X_test_freq['PaymentMethod'] = X_test_freq['PaymentMethod'].map(freq_map)



## 10. Target Encoding
Replaces a category with the mean value of the target variable for that category.


In [11]:

X_train_target = X_train.copy()
X_test_target = X_test.copy()

target_mean = pd.concat([X_train, y_train], axis=1).groupby('InternetService')['Churn'].mean()

X_train_target['InternetService'] = X_train_target['InternetService'].map(target_mean)
X_test_target['InternetService'] = X_test_target['InternetService'].map(target_mean)



## 12. Encoding Comparison

| Encoding | Dimensionality | Use Case |
|--------|--------|--------|
| Label Encoding | Low | Binary categories |
| One Hot Encoding | High | Nominal categories |
| Frequency Encoding | Low | High cardinality |
| Target Encoding | Low | Captures relationship with target |



## 13. Key Insights

- One Hot Encoding increases the number of features significantly.
- Label Encoding works best for binary categorical variables.
- Frequency Encoding avoids dimensionality explosion.
- Target Encoding can capture relationships with the target but must be used carefully to avoid data leakage.
